)<img src="https://www.epfl.ch/about/overview/wp-content/uploads/2020/07/logo-epfl-1024x576.png" style="padding-right:10px;width:140px;float:left"></td>
<h2 style="white-space: nowrap">Neural Signals and Signal Processing (NX-421)</h2>
<hr style="clear:both"></hr>


Welcome to the laboratory computers for the course "Neural signals and signal processing". Today, you will see how to preprocess fMRI data as well as play around with diffusion weighted imaging to compute a simple tractography.

In [ ]:
import sys
import os
import subprocess

#####################
# Import of utils.py functions
#####################
# Required to get utils.py and access its functions
notebook_dir = os.path.abspath("")
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.append(parent_dir)
sys.path.append('.')
from utils import loadFSL, FSLeyesServer, mkdir_no_exist, interactive_MCQ

####################
# DIPY_HOME should be set prior to import of dipy to make sure all downloads point to the right folder
####################
os.environ["DIPY_HOME"] = "/home/jovyan/Data"

#############################
# Loading fsl and freesurfer within Neurodesk
# You can find the list of available other modules by clicking on the "Softwares" tab on the left
#############################
import module as lmod
await lmod.purge(force=True)
await lmod.load('fsl/6.0.7.18')
await lmod.load('freesurfer/7.4.1')
await lmod.list()

####################
# Setup FSL path
####################
loadFSL()

###################
# Load all relevant libraries for the lab
##################
import fsl.wrappers
from fsl.wrappers import fslmaths

import mne_nirs
import nilearn
from nilearn.datasets import fetch_development_fmri

import mne
import mne_nirs

import xml.etree.ElementTree as ET
import os.path as op
import nibabel as nib
import glob

import openneuro
from mne.datasets import sample
from mne_bids import BIDSPath, read_raw_bids, print_dir_tree, make_report

# Useful imports to define the direct download function below
import requests
import urllib.request
from tqdm import tqdm


# FSL function wrappers which we will call from python directly
from fsl.wrappers import fast, bet
from fsl.wrappers.misc import fslroi
from fsl.wrappers import flirt

# General purpose imports to handle paths, files etc
import glob
import pandas as pd
import numpy as np
import json

!pip install --user -q jupyter-rfb

In [ ]:
%gui wx

In [ ]:
################
# Start FSLeyes (very neat tool to visualize MRI data of all sorts) within Python
################
fsleyesDisplay = FSLeyesServer()
fsleyesDisplay.show()

# Diffusion data and tractography generation

The preprocessing in DTI involves similar steps to what you saw in fMRI. We will thus tackle specific different steps to not repeat ourselves too much:
<center><img src="/files/Labs/Week03/imgs/DTI_preprocessing.png" width="600"/></center>
<p  style="text-align: center;"><i>Image from Son, Seong-Jin, Mansu Kim, and Hyunjin Park. "Imaging analysis of Parkinson’s disease patients using SPECT and tractography." Scientific reports 6.1 (2016): 1-11.</i></p>

In other words, we will have you look at an example to generate tractogram. Here, a tractogram will be generated using a deterministic algorithm EuDX.  
  
Diffusion tensor imaging (DTI) is one of the most popular MRI techniques to describe the orientation of white matter fibers in brain research. The process of fiber tracking is called tractography. It allows for a virtual dissection and three-dimensional representation of white matter tracts. 
While we could still use FSL for the task, we will have you use [DIPY](https://dipy.org/index.html), a Python package for computational neuroanatomy mainly focusing on diffusion MRI analysis.  
<br>
To generate a tractogram, we need to track the fibers, which is called fiber tracking.
<br>
Local fiber tracking is used to model white matter fibers by creating streamlines from local directional information. In order to perform local fiber tracking, you will apply the following three steps:
<p >
    <ol>
        <li style="font-size: 15px;">Extract directions from diffusion data</li> 
        <li style="font-size: 15px;">Identify when the tracking must stop</li>  
        <li style="font-size: 15px;">Select a set of locations from which to begin tracking</li>
    </ol>
</p>
Combining them will help you obtain a tractography reconstruction!

Ready? Let's go!

## 2.1. Load the data 

In [ ]:
from dipy.data import fetch_bundles_2_subjects, read_bundles_2_subjects
from dipy.core.gradients import gradient_table
from dipy.data import get_fnames
from dipy.io.gradients import read_bvals_bvecs
from dipy.io.image import load_nifti, load_nifti_data

hardi_fname, hardi_bval_fname, hardi_bvec_fname = get_fnames('stanford_hardi')
hardi_root_path = op.split(hardi_fname)[0]
label_fname = get_fnames('stanford_labels')

data, affine, hardi_img = load_nifti(hardi_fname, return_img=True)
labels = load_nifti_data(label_fname)
bvals, bvecs = read_bvals_bvecs(hardi_bval_fname, hardi_bvec_fname)
gtab = gradient_table(bvals, bvecs)

In [ ]:
print('data.shape: ',data.shape)
print('affine.shape: ',affine.shape)
print('hardi_img.shape: ',hardi_img.shape)

print('labels.shape: ',labels.shape)
print('bvals.shape: ',bvals.shape)
print('bvecs.shape: ',bvecs.shape)

In [ ]:
import os.path as op
from mne_bids import BIDSPath, read_raw_bids, print_dir_tree, make_report

print_dir_tree(hardi_root_path, max_depth=5)

## 2.2. Get the directions from the diffusion data set

### 2.2.1. Defining the white matter region.
Before all else, you will need to visualize the labels above. Run the following cell to load the labels on FSLeyes:

In [ ]:
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load('/home/jovyan/Data/stanford_hardi/aparc-reduced.nii.gz')
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[0]).cmap = 'brain_colours_spectrum'

Based on the values you read within, can you please fill in the cell below with the label corresponding to white matter?

In [ ]:
white_matter_value = ??? # Fill with the value you read in FSLeyes for white matter!

Great, let's now visualize the result of the mask, shall we? For this, let's generate the mask we obtained from above, using our best pal fslmaths ! 
What you should do is extract from the above file the white matter directly, using fslmaths (you can also use the pythonic approach using numpy vector operations seen in lab 2 if you want).

In [ ]:
???

That's nice, but we are missing something, aren't we? 
In the coronal slice, look in the middle. There is a region appearing in magenta/purple, but we wanted to include all white matter. Why is that? If you have on top, you'll see it has a different label: 2. This is because this region is white matter, but it is also a sagittal slice of the **corpus callosum**. So we need to do something a bit different. Can you think of a way to modify the above fslmaths command to include both the white matter and the slice of corpus callosum ? :)

In [ ]:
# ( We show the fslmaths approach, but you can still use the Python approach :) .)

lower_threshold = ??? # Select the lower bound to include both white matter and corpus callosum slice
upper_threshold = ??? # Select the upper bound to include both white matter and corpus callosum slice
output_name = op.join(hardi_root_path,'extracted_wm_complete')

subprocess.run(['fslmaths', label_fname, '-thr', str(lower_threshold), '-uthr', str(upper_threshold), '-bin', output_name])
fsleyesDisplay.load(output_name)

Beautiful! So you can see that these labels can be a bit tricky if you're not careful. Based on your above experience above, we will construct a mask in python directly. Please fill in the cell below the two values for:
- The white matter regions
- The corpus callosum slice

In [ ]:
corpus_callosum_slice_value = ??? # Fill with your value!
white_matter_value = ??? # Fill with your value !

total_white_matter = (labels == corpus_callosum_slice_value) | (labels == white_matter_value)

### 2.2.2. Actually extracting fiber orientations: the orientation distribution function
Okay, now we have a mask to define our fibers. The next cell will be used to estimate the orientation distribution function at each voxel. Before going any further, let's ask why this is necessary. In your opinion, in a single *voxel* how many orientations can we have?

In [ ]:
interactive_MCQ(3,4)

The issue can be summarized as resolving **intravoxel** fiber orientations of MR images.
To summarize these, we use an orientation distribution function, coined ODF.

The ODF is a function that describes how likely local water diffuses in a given orientation in 3D. 

We will not bore you with all mathematical details. Several models can be used to estimate ODFs, such as Constrained Spherical Deconvolution (CSD), Constant Solid Angle (CSA), Multi-Shell Multi-Tissue CSD( MSMT-CSD) and others. 

Here, we use CSD, which estimates the fiber orientation distribution by deconvolving the diffusion signal with the response of a single coherent fiber population (typically using the corpus callosum). This allows multiple fiber directions to be resolved within the same voxel.

The idea is that the resulting distribution provides useful information about the dominant fiber orientations and their relative strengths.

Let's now estimate the orientation distribution function of each voxel, using the CSD model. 

In [ ]:
from dipy.reconst.csdeconv import auto_response_ssst,ConstrainedSphericalDeconvModel
from dipy.data import default_sphere
from dipy.direction import peaks_from_model

# Single fiber response function: the measured signal of a single fiber
# sume: regions where there are single coherent fiber populations
# auto_response_ssst: calculate FA for a ROI of radii equal to roi_radii in the center of the volume
# and return the response function estimated in that region for the voxels with FA higher than 0.7
response, ratio = auto_response_ssst(gtab, data, roi_radii=10, fa_thr=0.7)

# Instantiate the Constrained Spherical Deconvolution model
csd_model = ConstrainedSphericalDeconvModel(gtab, response, sh_order_max=6)

Now that we have our model, the orientation of tract segments can be extracted, looking at the peaks in the model.

In [ ]:
csd_peaks = peaks_from_model(csd_model, data, default_sphere,
                             relative_peak_threshold=.5,
                             min_separation_angle=25,
                             mask=total_white_matter, npeaks=3)

Notice in the above cell the following line:
```python
csd_peaks = peaks_from_model(..., npeaks=3)
```

This means that really, we extract per voxel three peaks at most. This is an important assumption. Depending on your voxel size, you might want to pay attention to this number!

To confirm this, let's have a look at the extracted peak values: 

In [ ]:
csd_peaks.peak_values.shape

In [ ]:
# Force Vulkan
os.environ["WGPU_BACKEND_TYPE"] = "Vulkan"
# Tell PyGfx which GPU to use
os.environ["PYGFX_WGPU_ADAPTER_NAME"] = "NVIDIA L4-4Q"

from fury import actor, window, colormap

Knowing the MR dimensions, you can see that we indeed have three peaks per voxel. Great! Let's visualize it now! Please, note that the plot appearance might take a while!

In [ ]:
scene = window.Scene()

peaks_actor = actor.peaks_slicer(
    csd_peaks.peak_dirs,
    peak_values=csd_peaks.peak_values,
    affine=affine,
)

scene.add(peaks_actor)

showm = window.ShowManager(
    scene=scene,
    size=(900, 900),
    window_type="jupyter",
)

showm.start()

So as you can see, the orientations do map out to the expected directions of the fibers from the anatomy!

## 2.3. Set the stop criteria

Now, we need to setup our fiber tracking to stop it. What criterion should we use?
Well, we'll roughly use the idea that when we don't have enough evidence to know where a fiber could have gone, we stop tracking it.
In other words, if there are areas where the diffusion is totally unrestricted (goes in all directions), we have no clue as to where the fiber might continue. For this, we can threshold the tendency of our peaks to depend on a specific direction (anisotropy).<br>
More specifically, we will threshold the general fractional anisotropy (scalar measure between 0 and 1 with 0 → isotropic [e.g CSF, gray matter] and 1 → highly anisotropic [e.g white matter fiber bundles]) of our data to decide when we should stop.

In [ ]:
from dipy.tracking.stopping_criterion import ThresholdStoppingCriterion

stopping_criterion = ThresholdStoppingCriterion(csd_peaks.gfa, .25)

Let's visualize a slice! 

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

sli = csd_peaks.gfa.shape[2] // 2
plt.figure('GFA')
plt.subplot(1, 2, 1).set_axis_off()
plt.imshow(csd_peaks.gfa[:, :, sli].T, cmap='gray', origin='lower')

plt.subplot(1, 2, 2).set_axis_off()
plt.imshow((csd_peaks.gfa[:, :, sli] > 0.25).T, cmap='gray', origin='lower')

plt.savefig('gfa_tracking_mask.png')

## 2.4. Specify where to begin the fibers tracking

There are different ways to place seeds, ie starting points from which the fiber tracking is started. This depends on the pathways you might like to model! For example, if you only wanted to model the corpus callosum it would not be so interesting to place seeds in other regions of the brain. <br>
So that you understand what the output of the cell below will be, we must first explain what you'll extract in the cell below.<br>
The orientation of an image is described by its affine transformation, if you remember well. Let's call this affine $A$.
A seed point at the center of voxel $[i,j,k]$ will be represented as $[x,y,z, 1]= A \cdot [i,j,k,1]$<br>
In other words, you will get **coordinates in voxel space**. Note that there is one important assumption: the voxels here should be isotropic (the size of a voxel should be same along all directions).

In our specific case, we will start from a sagittal slice of the **corpus callosum**, the one with label 2 to be specific.
Please, create the mask (based on the labels above) to extract **only the slice of corpus callosum with label 2 as a mask**. You can refer to what we did above to do so. 

In [ ]:
from dipy.tracking import utils

seed_mask = ??? # Your code here to extract only the place of interest! 
seeds = utils.seeds_from_mask(seed_mask, affine, density=[2, 2, 2])

## 2.5. Bringing it all together and generating the streamlines
Let's see our ingredients:
- Seeds, generated above and starting from a slice of the corpus callosum
- A mask of regions where we should stop our fibers, based on anisotropy
- Peaks of ODF, at most five peaks per voxel

It remains now to combine all of these to bake so called streamlines (ie: fibers!). To do so, we will use the EuDX algorithm.

In [ ]:
from dipy.tracking.local_tracking import LocalTracking
from dipy.tracking.streamline import Streamlines

# Initialization of LocalTracking. The computation happens in the next step.
streamlines_generator = LocalTracking(csd_peaks, stopping_criterion, seeds,
                                      affine=affine, step_size=.5)
# Generate streamlines object
streamlines_t = Streamlines(streamlines_generator)

In [ ]:
streamlines_t

Beautiful! Let's now visualize our streamlines!
Remember that they represent **only lines that start from the corpus callosum**! 

In [ ]:
from fury import colormap

tract_actor = actor.line(
    streamlines_t,
    colors=colormap.line_colors(streamlines_t),
    material="basic",
    enable_picking=False,
)

scene = window.Scene()
scene.add(tract_actor)

showm = window.ShowManager(
    scene=scene,
    size=(900, 900),
    window_type="jupyter",
)

showm.start()

## 2.6. Store the streamlines into a trackvis file

What if we wanted to save the result as a file? Well, you can! For this, we need to save it to a special format, the TrackVis (.trk) format.

Remember: our goal was to generate the streamlines. It is these streamlines that we therefore want to save! :) 
Let's do it! 

In [ ]:
from dipy.io.stateful_tractogram import Space, StatefulTractogram
from dipy.io.streamline import save_tractogram, save_trk

# This is for the cc slice tractogram
sft = StatefulTractogram(streamlines_t, hardi_img, Space.RASMM)
save_trk(sft, "tractogram_EuDX.trk", streamlines_t)

If you want to visualize it all, you can activate [TrackVis](https://trackvis.org) and open the file from within or re-use `load_trk` from the dipy library.

## 2.7 Conclusions

Let's sum up what we've seen. For a successfull tractography generation, we need the following: 
<table>
    <tr>
        <th>Ingredient</th>
        <th>Role</th>
        <th>How is it created?</th>
    </tr>
    <tr>
        <td>Seeds</td>
        <td>Define starting point of tract propagation.</td>
        <td>Can be done randomly or according to some mask of interest</td>
    </tr>
    <tr>
        <td>Diffusion directions</td>
        <td>Define the local diffusion in a voxel, for all voxels of interest</td>
        <td>Can be done with CSD-ODF or other models such as e.g diffusion tensor</td>
    </tr>
    <tr>
        <td>Stopping criteria</td>
        <td>Defines where the tract continues or stops.</td>
        <td>Can be done based on anatomy, information of diffusion direction, combination of both...</td>
    </tr>
    <tr>
        <td>A tracking algorithm</td>
        <td>Combines all ingredients above to generate streamlines</td>
        <td>Line propagation techniques to grow from seed region, or probabilistic with a pdf of fiber orientations.</td>
    </tr>
</table> 


Each of the ingredients can be changed for a different flavour. You can explore [DIPY's tutorials](https://docs.dipy.org/stable/examples_built/index) to get an idea of the changes you can operate. Feel free to play around!

## 2.8 Remark: Connectivity analysis based on tractography 

By using the generated streamlines, we could analyze the brain connectivity, for example, which streamlines pass through or not pass through some regions of the brain, how many streamlines are connecting two ROI, etc. To do this, it would be better if we could create a tractography with seeds spaning the entire white matter. Due to RAM concern, we will not do it here. But please feel free to explore it if you are interested! You will find some useful tutorials [here](https://docs.dipy.org/stable/examples_built/index#streamlines-analysis-and-connectivity) 
This ends this short tractography tutorial! Again, do not hesitate to explore more on DIPY's website if you're interested :)
We strongly encourage you to use TrackVis for visualization of tractograms as it's really made for it and is much more intuitive to use for this purpose than FSLeyes.

<div class="alert alert-success">
<p><b>🎉 You've reached the end of this week's notebook! Congratulations! 🎉 </b></p>
</div>